# Preprocessing Ulasan Wisata



**TONDI — Toba Observatory for Natural-language Destination Intelligence**

Notebook ini mendokumentasikan pipeline preprocessing ulasan wisata Danau Toba.

**Langkah-langkah:**
1. Load dataset
2. Lowercase
3. Hapus URL
4. Hapus emoji
5. Hapus angka
6. Hapus tanda baca
7. Hapus spasi berlebih
8. Normalisasi slang Indonesia
9. Normalisasi kata Batak
10. Stopword removal
11. Tokenisasi
12. Pelabelan sentimen & topik


## Persiapan Lingkungan


In [ ]:
# Install dependensi (jalankan sekali saja)
!pip install nltk folium openpyxl


In [ ]:
# Download stopwords NLTK
import nltk
nltk.download('stopwords', quiet=True)
print('Stopwords siap!')


## 1. Import Library


In [ ]:
import pandas as pd
import re
import string
import os

# Preprocessing
from clean_review import clean_review, tokenize
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "labeling"))
from labeling import label_sentimen, label_topik
import folium

print("Library berhasil diimpor!")

## 2. Load Dataset


Dataset ulasan wisata Danau Toba yang sudah dibersihkan.

In [ ]:
path = "../../dataset/processed/review_clean.xlsx"
df = pd.read_excel(path)
print(f"Jumlah ulasan: {len(df)} baris")
print(f"Kolom: {list(df.columns)}")
df[["review-text", "review_clean"]].head()

## 3. Contoh Data


In [ ]:
for i in range(5):
    asli = str(df["review-text"].iloc[i])[:100]
    bersih = str(df["review_clean"].iloc[i])[:100]
    print(f"Asli   : {asli}")
    print(f"Bersih : {bersih}")
    print()

## 4. Demo Preprocessing


Menunjukkan cara kerja setiap fungsi pembersihan secara manual.

In [ ]:
sample = "Tempatnya KEREN banget! Kunjungi https://example.com untuk info udah 2 kali"
print("Asli    :", sample)
print("Hasil   :", clean_review(sample))
print("Token   :", tokenize(clean_review(sample)))

### Normalisasi Slang


In [ ]:
sample = "Tp tempatnya bener2 keren bgt, gak ada sampah. recommended!"
print("Asli    :", sample)
print("Hasil   :", clean_review(sample))
print("Token   :", tokenize(clean_review(sample)))

### Normalisasi Batak


In [ ]:
sample = "Horas! Danau toba di samosir sangat indah, banyak ulos dan sigale-gale."
print("Asli    :", sample)
print("Hasil   :", clean_review(sample))
print("Token   :", tokenize(clean_review(sample)))

## 5. Pelabelan Sentimen


Menggunakan aturan rule-based dengan keyword positif/negatif.

In [ ]:
samples = [
    "pantainya bersih indah recommended",
    "banyak sampah kotor sekali",
    "parkir mahal tidak ramah",
    "danau toba samosir",
    "pungli merajalela"
]
for s in samples:
    print(f"{s:45} -> {label_sentimen(s)}")

## 6. Klasifikasi Topik



8 kategori topik:
- Kebersihan, Pungli, Harga, Layanan
- Fasilitas, Akses, Parkir, Keamanan


In [ ]:
samples = [
    "sampah berserakan bau tidak sedap",
    "pungli parkir meresahkan",
    "tiket masuk mahal sekali",
    "pelayanan ramah petugas membantu",
    "toilet bersih dan mushola nyaman",
    "jalan berlubang akses sulit",
    "lahan parkir sempit",
    "gelap tidak ada lampu tidak aman"
]
for s in samples:
    print(f"{s:45} -> {label_topik(s):12} | {label_sentimen(s)}")

## 7. Statistik Dataset


Distribusi sentimen dan topik pada seluruh dataset.

In [ ]:
# Load hasil pelabelan
df_label = pd.read_excel("../../dataset/processed/review_labeled.xlsx")

print("=" * 40)
print("DISTRIBUSI SENTIMEN")
print("=" * 40)
for label, count in df_label["sentiment"].value_counts().items():
    print(f"  {label:8}: {count:5} ({count/len(df_label)*100:.1f}%)")

print()
print("=" * 40)
print("DISTRIBUSI TOPIK")
print("=" * 40)
for label, count in df_label["topic"].value_counts().items():
    print(f"  {label:12}: {count:5} ({count/len(df_label)*100:.1f}%)")

## 8. Visualisasi Peta Destinasi


Peta interaktif destinasi wisata Danau Toba menggunakan Folium.

In [ ]:
df_dest = pd.read_excel("../../dataset/processed/destination.xlsx")

# Jika destination.xlsx kosong, baca dari raw
if len(df_dest) == 0:
    path_raw = "../../dataset/raw/Dataset HackathonTourism - IT DEL.xlsx"
    dfs = []
    for sheet, cat in [("wisata-metadata", "Wisata"), ("hotel-metadata", "Hotel"), ("resto-metadata", "Resto")]:
        d = pd.read_excel(path_raw, sheet_name=sheet)
        for _, r in d.iterrows():
            if pd.notna(r.get("lat-long")) and pd.notna(r.get("place-name")):
                parts = str(r["lat-long"]).split(",")
                if len(parts) == 2:
                    dfs.append({"place": r["place-name"], "cat": cat, "lat": float(parts[0]), "lng": float(parts[1])})
    df_dest = pd.DataFrame(dfs)

peta = folium.Map(location=[2.5, 98.9], zoom_start=10)
for _, r in df_dest.iterrows():
    warna = {"Wisata": "green", "Hotel": "blue", "Resto": "red"}.get(r["cat"], "gray")
    folium.Marker([r["lat"], r["lng"]], popup=r["place"], tooltip=r["cat"], icon=folium.Icon(color=warna)).add_to(peta)
peta


## 9. Kesimpulan



Pipeline preprocessing telah berhasil diterapkan pada **12.691** ulasan wisata Danau Toba dengan hasil:

- **30.1%** reduksi karakter (teks dibersihkan)
- **26.6%** ulasan berlabel **Positif**
- **3.6%** ulasan berlabel **Negatif**
- **69.8%** ulasan berlabel **Netral**

8 topik terdeteksi: Kebersihan, Pungli, Harga, Layanan, Fasilitas, Akses, Parkir, Keamanan.

Visualisasi geospasial menampilkan **323** destinasi (wisata, hotel, resto) di kawasan Danau Toba.
